In [ ]:
import torch
import torch.nn as nn
from torchvision import models
 
 
class DeepfakeVGG16(nn.Module):
    """
    VGG-16 fine-tuned for binary deepfake detection.
 
    Strategy:
      - Load ImageNet-pretrained VGG-16 weights.
      - Freeze early convolutional blocks (features[0:17]) to retain low-level
        edge / texture features while reducing training cost.
      - Replace the original 1000-class classifier with a lightweight binary head
        that includes dropout for regularisation.
    """
 
    def __init__(self, freeze_features: bool = True, dropout_p: float = 0.5):
        super().__init__()
 
        # ── Backbone ──────────────────────────────────────────────────────────
        backbone = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
 
        self.features = backbone.features          # Conv blocks 1-5
        self.avgpool  = backbone.avgpool           # Adaptive 7×7 pool
 
        # Optionally freeze early feature layers (conv1 + conv2 blocks)
        if freeze_features:
            for param in self.features[:17].parameters():
                param.requires_grad = False
 
        # ── Custom binary classifier ───────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(4096, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(1024, 1),          # Single logit → BCEWithLogitsLoss
        )
 
        self._initialise_classifier()
 
    # ── Weight init ───────────────────────────────────────────────────────────
    def _initialise_classifier(self):
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                nn.init.zeros_(layer.bias)
 
    # ── Forward ───────────────────────────────────────────────────────────────
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x                         # Raw logits; apply sigmoid for inference
 
    # ── Convenience helpers ───────────────────────────────────────────────────
    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        """Returns probability that the image is a deepfake (0 = real, 1 = fake)."""
        with torch.no_grad():
            return torch.sigmoid(self.forward(x))
 
    def predict(self, x: torch.Tensor, threshold: float = 0.5) -> torch.Tensor:
        """Returns binary label: 1 = deepfake, 0 = real."""
        return (self.predict_proba(x) >= threshold).long()